# Phase 0 — Verify the original pretrained SEN2SR model

Goal: confirm that the official **SEN2SRLite / NonReference_RGBN_x4** model loads and runs correctly on this machine before any experiment, and record how it actually behaves.

Checks:
1. Download / load the official model package.
2. Hard-constraint mask: shape and filter profile.
3. Official example pair: SEN2SR vs bicubic, CPU vs MPS, consistency with the LR input.
4. Which CNN blocks actually influence the output.
5. Differentiable `train_mode=True` path reproduces the inference model (needed for fine-tuning later).
6. 121×121 SEN2NAIP tiles: `predict_large` behaviour and choice of padding.
7. One real dataset tile end to end.
8. Inference speed.

## Settings

In [1]:
MODEL_NAME = "SEN2SRLite_NonReference_RGBN_x4"
MLM_URL = "https://huggingface.co/tacofoundation/sen2sr/resolve/main/SEN2SRLite/NonReference_RGBN_x4/mlm.json"
MODEL_DIR = "model/SEN2SRLite_RGBN_x4"   # git-ignored
DATA_ROOT = "sen2naip/cross-sensor"       # git-ignored

SCALE = 4
TILE_SIZE = 128             # the published hard-constraint mask is fixed at 512x512 -> 128 px LR tiles
REFLECTANCE_DIVISOR = 10_000
PAD_MODE = "replicate"      # chosen in check 6 below

In [2]:
import hashlib, time, warnings
from pathlib import Path

import mlstac
import numpy as np
import rasterio
import safetensors.torch
import sen2sr
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_dir = Path(MODEL_DIR)
print("torch", torch.__version__, "| sen2sr", sen2sr.__version__, "| device", DEVICE)

torch 2.14.1 | sen2sr 0.8.5 | device mps


## 1. Download and load the official model

In [3]:
if not (model_dir / "model.safetensor").exists():
    mlstac.download(file=MLM_URL, output_dir=str(model_dir))

for f in sorted(model_dir.glob("*.safetensor")):
    print(f"{f.name:28s} sha256={hashlib.sha256(f.read_bytes()).hexdigest()[:16]}...")

def load_sen2sr(device):
    """Official loader: CNN (inference mode, frozen) + hard constraint."""
    return mlstac.load(str(model_dir)).compiled_model(device=str(device)).eval()

def psnr(a, b, data_range=1.0):
    mse = F.mse_loss(a.float().cpu(), b.float().cpu()).item()
    return 10 * np.log10(data_range**2 / mse)

model = load_sen2sr(DEVICE)
model_cpu = load_sen2sr("cpu")
print(model.sr_model.__class__.__name__, "| trainable-looking params:",
      sum(p.numel() for p in model.sr_model.parameters()))

example_data.safetensor      sha256=c895c7da8a8d4888...
hard_constraint.safetensor   sha256=fbad981519066387...
model.safetensor             sha256=479aa796d5068d0b...
CNNSR | trainable-looking params: 572336


## 2. Hard-constraint mask

The forward pass is `sr = clamp(CNN(x), min=0)` → `HardConstraint(x, sr)`, which keeps the bicubic-upsampled LR spectrum where the mask is 1 and the CNN spectrum where it is 0. Here we check what the mask really looks like.

In [4]:
mask = safetensors.torch.load_file(model_dir / "hard_constraint.safetensor")["weights"]
h, w = mask.shape
yy, xx = torch.meshgrid(torch.arange(h) - h // 2, torch.arange(w) - w // 2, indexing="ij")
d2 = (yy**2 + xx**2).float()

# Gaussian fit: -ln(m) = d^2 / (2 sigma^2), using well-conditioned values
sel = (mask > 1e-3) & (mask < 0.999)
sigma = float(torch.sqrt(1 / (2 * (d2[sel] * -torch.log(mask[sel])).sum() / (d2[sel] ** 2).sum())))
fit_err = float((torch.exp(-d2 / (2 * sigma**2)) - mask).abs().max())

nyq = min(h, w) // 2 // SCALE   # LR Nyquist radius in the 512x512 spectrum
row = mask[h // 2, w // 2:]
print(f"shape {tuple(mask.shape)} | Gaussian sigma = {sigma:.2f} (max fit error {fit_err:.1e})")
print(f"LR Nyquist radius = {nyq}")
for r in [nyq // 2, nyq, int(1.5 * nyq)]:
    print(f"  mask at r={r:3d}: {row[r]:.3f}  (share of the LR/bicubic spectrum kept)")

shape (512, 512) | Gaussian sigma = 35.00 (max fit error 6.0e-08)
LR Nyquist radius = 64
  mask at r= 32: 0.658  (share of the LR/bicubic spectrum kept)
  mask at r= 64: 0.188  (share of the LR/bicubic spectrum kept)
  mask at r= 96: 0.023  (share of the LR/bicubic spectrum kept)


## 3. Official example pair

In [5]:
lr_ex, hr_ex = mlstac.load(str(model_dir)).example_data()
with torch.no_grad():
    sr_cpu = model_cpu(lr_ex)
    sr_dev = model(lr_ex.to(DEVICE)).cpu()
bicubic = F.interpolate(lr_ex, scale_factor=SCALE, mode="bicubic", antialias=True)

print("LR", tuple(lr_ex.shape), "HR", tuple(hr_ex.shape), "| LR range", round(lr_ex.min().item(), 4), "-", round(lr_ex.max().item(), 4))
print("CPU vs", DEVICE, "max abs diff:", f"{(sr_cpu - sr_dev).abs().max().item():.1e}", "| NaN:", torch.isnan(sr_cpu).any().item())
print(f"PSNR vs HR      bicubic {psnr(bicubic, hr_ex):.2f} dB | SEN2SR {psnr(sr_cpu, hr_ex):.2f} dB")
print(f"LR consistency  bicubic {psnr(F.avg_pool2d(bicubic, SCALE), lr_ex):.2f} dB | SEN2SR {psnr(F.avg_pool2d(sr_cpu, SCALE), lr_ex):.2f} dB")

LR (1, 4, 128, 128) HR (1, 4, 512, 512) | LR range 0.0187 - 0.9984
CPU vs mps max abs diff: 9.5e-07 | NaN: False
PSNR vs HR      bicubic 32.99 dB | SEN2SR 33.35 dB
LR consistency  bicubic 48.87 dB | SEN2SR 52.30 dB


## 4. Which CNN blocks influence the output?

Each block's weights are perturbed in turn; a change of exactly 0 means the block is not used.

In [6]:
cnn = model_cpu.sr_model
torch.manual_seed(0)
with torch.no_grad():
    base = cnn(lr_ex)
    for i, block in enumerate(cnn.blocks):
        saved = {k: v.clone() for k, v in block.state_dict().items()}
        for p in block.parameters():
            p.add_(torch.randn_like(p) * 0.1)
        print(f"block {i}: max output change {(cnn(lr_ex) - base).abs().max().item():.4f}")
        block.load_state_dict(saved)

block 0: max output change 1.3183
block 1: max output change 0.0000
block 2: max output change 0.0000
block 3: max output change 0.0000
block 4: max output change 0.0000
block 5: max output change 0.7822


## 5. Differentiable training path

In inference mode the conv weights are re-fused (and detached) on every forward pass, so gradients cannot flow. The `train_mode=True` branch should give the same output.

In [7]:
from sen2sr.models.opensr_baseline.cnn import CNNSR

train_cnn = CNNSR(4, 4, 24, 4, True, True, 6)
train_cnn.load_state_dict(safetensors.torch.load_file(model_dir / "model.safetensor"))
with torch.no_grad():
    print("eval vs train_mode max abs diff:", f"{(cnn(lr_ex) - train_cnn(lr_ex)).abs().max().item():.1e}")

eval vs train_mode max abs diff: 1.1e-06


## 6. 121×121 tiles: `predict_large` and padding

SEN2NAIP LR tiles are 121×121. Reference: the same area predicted with full 128 px context.

In [8]:
n = 121
try:
    sen2sr.predict_large(X=lr_ex[0, :, :n, :n], model=model_cpu, overlap=32)
    print("predict_large: ok")
except RuntimeError as e:
    print("predict_large fails:", e)

@torch.no_grad()
def super_resolve(model, lr, pad_mode=PAD_MODE):
    """SR for [B, C, H, W] reflectance tiles with H, W <= TILE_SIZE: pad bottom/right, run, crop."""
    _, _, h, w = lr.shape
    if h > TILE_SIZE or w > TILE_SIZE:
        raise ValueError(f"Tile {h}x{w} exceeds {TILE_SIZE} px; large-image tiling is not implemented yet.")
    device = next(model.parameters()).device
    x = F.pad(lr, (0, TILE_SIZE - w, 0, TILE_SIZE - h), mode=pad_mode)
    return model(x.to(device))[..., : h * SCALE, : w * SCALE].cpu()

with torch.no_grad():
    ref = model_cpu(lr_ex)[..., : n * SCALE, : n * SCALE]
edge = slice(n * SCALE - 8 * SCALE, None)   # last 8 LR px next to the padding
for mode in ["reflect", "replicate", "constant"]:
    sr = super_resolve(model_cpu, lr_ex[..., :n, :n], mode)
    print(f"{mode:10s} PSNR vs full context: whole {psnr(sr, ref):.2f} dB | bottom edge {psnr(sr[..., edge, :], ref[..., edge, :]):.2f} dB")

  0%|          | 0/1 [00:00<?, ?it/s]

predict_large fails: The size of tensor a (484) must match the size of tensor b (512) at non-singleton dimension 3
reflect    PSNR vs full context: whole 57.46 dB | bottom edge 51.52 dB
replicate  PSNR vs full context: whole 59.85 dB | bottom edge 55.53 dB


constant   PSNR vs full context: whole 41.70 dB | bottom edge 33.14 dB


## 7. One real SEN2NAIP tile end to end

In [9]:
with rasterio.open(f"{DATA_ROOT}/ROI_0000/lr.tif") as src:
    lr_tile = src.read().astype(np.float32)
x = torch.from_numpy(np.clip(lr_tile / REFLECTANCE_DIVISOR, 0, None))[None]
sr = super_resolve(model, x)
print(tuple(x.shape), "->", tuple(sr.shape), "| range", round(sr.min().item(), 4), "-", round(sr.max().item(), 4), "| NaN", torch.isnan(sr).any().item())
print("band means LR", x[0].mean((1, 2)).numpy().round(4))
print("band means SR", sr[0].mean((1, 2)).numpy().round(4))

(1, 4, 121, 121) -> (1, 4, 484, 484) | range 0.0068 - 0.2572 | NaN False
band means LR [0.0958 0.0765 0.0496 0.1878]
band means SR [0.0958 0.0765 0.0496 0.1878]


## 8. Inference speed

In [10]:
def ms_per_tile(m, device, batch=16, reps=5):
    x = torch.rand(batch, 4, TILE_SIZE, TILE_SIZE, device=device)
    sync = torch.mps.synchronize if torch.device(device).type == "mps" else (lambda: None)
    with torch.no_grad():
        m(x); sync()
        t0 = time.perf_counter()
        for _ in range(reps):
            m(x)
        sync()
    return (time.perf_counter() - t0) / (reps * batch) * 1000

print(f"{DEVICE}: {ms_per_tile(model, DEVICE):.1f} ms per 128 px tile")
print(f"cpu: {ms_per_tile(model_cpu, 'cpu'):.1f} ms per 128 px tile")

mps: 9.6 ms per 128 px tile
cpu: 53.6 ms per 128 px tile


## Findings

- **Mask:** an exact Gaussian low-pass (σ = 35 in the 512×512 spectrum). It is a *soft* blend, not a cutoff at the LR Nyquist radius (64): ≈ 0.66 of the LR spectrum is kept at half-Nyquist, ≈ 0.19 at Nyquist, ≈ 0.02 at 1.5× Nyquist.
- **Example pair:** SEN2SR beats bicubic modestly; CPU and MPS agree to ~1e-6. LR consistency is high but not exact, because of the soft mask.
- **Blocks 1–4 are unused** (every block receives the same input; only blocks 0 and 5 reach the output). Kept as published; LoRA should target only the modules that matter.
- **`train_mode=True`** reproduces inference → fine-tuning can use it without modifying SEN2SR.
- **`predict_large` fails on 121 px tiles**; `super_resolve` pads to 128 with **replicate** padding (closest to full context) and crops back.
- **Speed:** the whole dataset runs in well under a minute on MPS.

## Tests

Automated checks of everything above. Each test prints PASS or raises an `AssertionError`.

In [11]:
EXPECTED_SHA256 = {
    "example_data.safetensor": "c895c7da8a8d48882b73a2a1955e4260714b97540eea290229a284d73f129985",
    "hard_constraint.safetensor": "fbad981519066387c413ead1d6af7ef3e0d2947c34147ba90163fc79ae539239",
    "model.safetensor": "479aa796d5068d0b1206118ccbca27bd3223df0214db1a9b31a1e18349ed1c7e",
}
results = {}

def test(name):
    def wrap(fn):
        try:
            fn()
            results[name] = "PASS"
        except Exception as e:
            results[name] = f"FAIL: {type(e).__name__}: {e}"
        print(f"{results[name][:4]}  {name}" + ("" if results[name] == "PASS" else f"  -> {results[name]}"))
    return wrap

@test("weights match recorded SHA-256 hashes")
def _():
    for name, sha in EXPECTED_SHA256.items():
        assert hashlib.sha256((model_dir / name).read_bytes()).hexdigest() == sha, name

@test("mask is a 512x512 radially symmetric Gaussian, sigma = 35, values in [0, 1]")
def _():
    assert mask.shape == (TILE_SIZE * SCALE, TILE_SIZE * SCALE)
    assert 0 <= mask.min() and mask.max() <= 1 and mask[h // 2, w // 2] == 1
    assert abs(sigma - 35) < 0.01 and fit_err < 1e-5
    assert torch.allclose(mask[h // 2, w // 2 + 1:], mask[h // 2, 1:w // 2].flip(0), atol=1e-6)

@test("example output: shape, finite, CPU == MPS")
def _():
    assert sr_cpu.shape == hr_ex.shape and torch.isfinite(sr_cpu).all()
    assert (sr_cpu - sr_dev).abs().max() < 1e-4

@test("inference is deterministic")
def _():
    with torch.no_grad():
        assert torch.equal(model_cpu(lr_ex), model_cpu(lr_ex))

@test("SEN2SR beats bicubic on the example pair (PSNR and LR consistency)")
def _():
    assert psnr(sr_cpu, hr_ex) > psnr(bicubic, hr_ex)
    assert psnr(F.avg_pool2d(sr_cpu, SCALE), lr_ex) > psnr(F.avg_pool2d(bicubic, SCALE), lr_ex)

@test("blocks 1-4 have no effect, blocks 0 and 5 do; model restored after perturbation")
def _():
    torch.manual_seed(1)
    with torch.no_grad():
        base = cnn(lr_ex)
        for i, block in enumerate(cnn.blocks):
            saved = {k: v.clone() for k, v in block.state_dict().items()}
            for p in block.parameters():
                p.add_(torch.randn_like(p) * 0.1)
            change = (cnn(lr_ex) - base).abs().max().item()
            block.load_state_dict(saved)
            assert (change == 0) == (i in (1, 2, 3, 4)), (i, change)
        assert torch.equal(cnn(lr_ex), base)

@test("train_mode=True reproduces the inference CNN")
def _():
    with torch.no_grad():
        assert (cnn(lr_ex) - train_cnn(lr_ex)).abs().max() < 1e-4

@test("predict_large fails on 121 px tiles (reason super_resolve exists)")
def _():
    try:
        sen2sr.predict_large(X=lr_ex[0, :, :121, :121], model=model_cpu, overlap=32)
    except RuntimeError:
        return
    raise AssertionError("predict_large unexpectedly succeeded; padding may no longer be needed")

@test("super_resolve: output shapes for square, non-square and full tiles")
def _():
    for hh, ww in [(121, 121), (128, 128), (64, 100), (1, 1)]:
        out = super_resolve(model, lr_ex[..., :hh, :ww])
        assert out.shape == (1, 4, hh * SCALE, ww * SCALE), (hh, ww, out.shape)

@test("super_resolve: 128 px input equals the raw model output")
def _():
    with torch.no_grad():
        assert (super_resolve(model_cpu, lr_ex) - model_cpu(lr_ex)).abs().max() < 1e-5

@test("super_resolve: batch result equals one-by-one results")
def _():
    batch = torch.cat([lr_ex[..., :121, :121], lr_ex[..., 7:, 7:]])
    out = super_resolve(model, batch)
    for i in range(2):
        assert (out[i] - super_resolve(model, batch[i:i + 1])[0]).abs().max() < 1e-4

@test("super_resolve: rejects tiles larger than 128 px")
def _():
    try:
        super_resolve(model, torch.rand(1, 4, 129, 64))
    except ValueError:
        return
    raise AssertionError("no error for a 129 px tile")

@test("20 real SEN2NAIP tiles: 484x484, finite, band means preserved")
def _():
    rng = np.random.default_rng(0)
    rois = sorted(Path(DATA_ROOT).glob("ROI_*"))
    picks = [rois[i] for i in rng.choice(len(rois), 20, replace=False)]
    tiles = []
    for roi in picks:
        with rasterio.open(roi / "lr.tif") as src:
            tiles.append(np.clip(src.read().astype(np.float32) / REFLECTANCE_DIVISOR, 0, None))
    x = torch.from_numpy(np.stack(tiles))
    out = super_resolve(model, x)
    assert out.shape == (20, 4, 484, 484) and torch.isfinite(out).all()
    rel = ((out.mean((2, 3)) - x.mean((2, 3))).abs() / x.mean((2, 3))).max().item()
    assert rel < 0.01, f"band mean drift {rel:.4f}"

failed = [k for k, v in results.items() if v != "PASS"]
print(f"\n{len(results) - len(failed)}/{len(results)} tests passed")
assert not failed, failed

PASS  weights match recorded SHA-256 hashes
PASS  mask is a 512x512 radially symmetric Gaussian, sigma = 35, values in [0, 1]
PASS  example output: shape, finite, CPU == MPS
PASS  inference is deterministic
PASS  SEN2SR beats bicubic on the example pair (PSNR and LR consistency)
PASS  blocks 1-4 have no effect, blocks 0 and 5 do; model restored after perturbation
PASS  train_mode=True reproduces the inference CNN


  0%|          | 0/1 [00:00<?, ?it/s]

PASS  predict_large fails on 121 px tiles (reason super_resolve exists)


PASS  super_resolve: output shapes for square, non-square and full tiles
PASS  super_resolve: 128 px input equals the raw model output
PASS  super_resolve: batch result equals one-by-one results
PASS  super_resolve: rejects tiles larger than 128 px
PASS  20 real SEN2NAIP tiles: 484x484, finite, band means preserved

13/13 tests passed
